In [1]:
! pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort

In [1]:
import sys, os
import matplotlib
import time
import pandas as pd
import numpy
import ast
import json
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from sklearn.neighbors import KNeighborsClassifier as knnbase
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.naive_bayes import MultinomialNB as mnb
from sklearn.linear_model import LogisticRegression as LR
from sklearn.naive_bayes import GaussianNB as GNB

from autogluon.tabular import TabularDataset
from autogluon.tabular import TabularPredictor as task
from autogluon.core.utils import infer_problem_type

from sklearn.model_selection import GridSearchCV as GSCV
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler as MMS
from sklearn.preprocessing import StandardScaler as SS

from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import multilabel_confusion_matrix as ML_matrix
from sklearn.metrics import precision_recall_fscore_support as score_multi
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from pickle import load, dump

In [2]:
# -------------------------------- HELPERS ------------------------------------------ #
def split_df(Xdata, labels, testsplit=0.3):
	Xtrain,Xtest,ytrain,ytest = train_test_split(Xdata,labels,test_size=testsplit)
	return Xtrain, Xtest, ytrain, ytest

# Rescale values to fit in a range; default: 0-1
def normalize(Xtrain, Xtest):
	scaler = MMS(feature_range=(0,1))
	Xtrainscaled = scaler.fit_transform(Xtrain)
	Xtestscaled = scaler.transform(Xtest)
	return Xtrainscaled, Xtestscaled

# Scale values such that mean = 0, std dev. = 1; Ensures robustness for new data.
def standardize(Xtrain, Xtest):
	ss = SS()
	Xtrainscaled = ss.fit_transform(Xtrain)
	Xtestscaled = ss.transform(Xtest)
	return Xtrainscaled, Xtestscaled, ss

def micro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='micro')
	recall = recall_score(y_test_multilabel, predictions, average='micro')
	f1 = f1_score(y_test_multilabel, predictions, average='micro')

	print("::Micro-average::")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	print("\n\n")
	return precision, recall, f1

def macro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='macro')
	recall = recall_score(y_test_multilabel, predictions, average='macro')
	f1 = f1_score(y_test_multilabel, predictions, average='macro')

	print("\nMacro-average: ")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	return

def per_class_dist(ytest, ypred, classorder):
	perclass = classification_report(ytest, ypred)
	print("Per class classification report: ", perclass)
	precision, recall, fscore, support = score_multi(ytest, ypred, average="micro")
	print('micro-precision: {}'.format(precision))
	print('micro-recall: {}'.format(recall))
	print('micro-fscore: {}'.format(fscore))
	print('support: {}'.format(support))
	#print(classorder)
	return

def output_avg(total, ag_res1, ag_res2, fimp1, fimp2, auto_cmatrix, perf, auc_score, ff):
	print(auto_cmatrix)
	ff.write("-----------------Autogluon----------------\n")
	ff.write("Best model confusion matrix: \n")
	[tn,fp,fn,tp] = auto_cmatrix
	fpr = float(fp/(fp+tn)*100)
	ff.write("TN: "+str(tn)+" FP: "+str(fp)+" FN: "+str(fn)+" TP: "+str(tp)+"\n")
	ff.write("::Model performance on test data::\n")
	ff.write("AUC Score: "+str(auc_score)+"\n")
	ff.write("FPR: "+str(fpr)+"\n")
	ff.write("Performance summary: "+str(perf)+" \n")
	ff.write(str(ag_res1))
	if not fimp1 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp1.head(20))+"\n")
	ff.write("\n::Stacking & Weighted Ensembling of Models::\n")
	ff.write(str(ag_res2))
	if not fimp2 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp2.head(20))+"\n")
	ff.write("--------------------------------------------\n")
	ff.close()
	return

In [3]:
def test_main(xtest, ytest, pred, testdf, traindf, calcftimpo=False):
	modelperf = pred.leaderboard(testdf, silent= True)
	print("[*]Model performance breakdown on Test data:")
	print(modelperf)
	ypred = pred.predict(xtest)
	ypredproba = pred.predict_proba(xtest)
	perf = pred.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	print("[*]Confidence in predictions:\n")
	print(pd.DataFrame(ypredproba, columns=pred.class_labels))
	# Each model score
	print("Perf: ", perf)
	print("Getting confusion matrix.....")
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print(cmatrix)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	print("AUC score for best model: ", auc_score)

	if calcftimpo:
		ftimpo = None
		ftimpo = pred.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	else:
		ftimpo = None
	bestmodel = pred.model_best
	return modelperf, ftimpo, cmatrix, ypredproba, bestmodel, perf, auc_score

def test_stack(xtest, ytest, predstack, testdf, traindf, calcftimpo=False):
	ypred = predstack.predict(xtest)
	ypredproba = predstack.predict_proba(xtest)
	perf = predstack.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	test_perf = predstack.leaderboard(testdf, silent=True)
	print("$$$$$$$$ RESULT STACKING $$$$$$$$\n", test_perf)
	ftimpo = None
	if calcftimpo:
		ftimpo = predstack.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print("Confusion matrix stacked: ", cmatrix)
	print("AUC using stacked model: ", auc_score)
	return test_perf, ftimpo, cmatrix, auc_score

In [4]:
def train_main(dataf, targetcol):
	agdir = os.getcwd()+'/AGmodels/'
	#dir = agdir+"/"+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir):
		os.system("mkdir "+agdir)

	predictor = task(label=targetcol, path=agdir, eval_metric='balanced_accuracy').fit(dataf, verbosity=4)
	return predictor

# Multi layer stacking takes predictions of base models and feeds to stack models
# AG will auto choose k= 10 fold cv, n=20 bagging repeats,
# L: 2 layers of models in stack followed by weighted-ensemble (higher weight for the model that performed well);
# Aggregate model predictions based on model weights and produce final prediction
def train_multilayerstacking(traindf, target):
	agdir_stack = os.getcwd()+'/AGmodels/stacked/' #+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir_stack):
		os.system("mkdir "+agdir_stack)
	predstack = task(label=target, path=agdir_stack, eval_metric='balanced_accuracy').fit(train_data= traindf, auto_stack=True, verbosity=3)
	return predstack

def main_ag(traindf, testdf, targetcol):
	# Displaying dataframe info
	x_test = testdf.iloc[:,:-1].copy()
	y_test = testdf.iloc[:,-1].copy()
	proxy_train = traindf[traindf['target'] == 1].shape
	proxy_test = testdf[testdf['target'] == 1].shape
	normal_train = traindf[traindf['target'] == 0].shape
	normal_test = testdf[testdf['target'] == 0].shape
	print("Train df (w/ target): ",traindf, traindf.shape)
	print("Train mal: ", proxy_train, "Train ben: ", normal_train)
	print("Test df (w/ target): ",testdf, testdf.shape)
	print("Test mal: ", proxy_test, "Test ben: ", normal_test)
	time.sleep(2)

  # Training binary classifiers: 8 base models, 2 DL models
	predictor = train_main(traindf, targetcol)
	predstack = train_multilayerstacking(traindf, targetcol)

	# Testing binary classifiers
	print("###################~Testing Trained Models (30% PCAPs)~############################")
	res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score = test_main(x_test, y_test, predictor, testdf, traindf)
	# Uncomment for test results with feature importance (longer run time)
	##res1, fimp1, cmatrix, ytest, ypred_proba, bestmodel, perf, auc_score = test_main(Xtest, ytest, predictor, testdf, traindf, True)

	print("####################Stacking & Weighted Ensemble Testing###########################")
	res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_test, y_test, predstack, testdf, traindf)
	# With feature importance
	##res2, fimp2, cmatrixstacked, aucstacked = test_stack(Xtest, ytest, predstack, testdf, traindf, True)

	return [res1, res2, fimp1, fimp2, cmatrix, bestmodel, perf, auc_score]

In [5]:
foldtotal = 10
relayed_fts_path = '/content/features_lim_relayed.csv'

background_fts_path = '/content/features_lim_background.csv'


relayed_feats=pd.read_csv(relayed_fts_path)
#gw_feats_high=pd.read_csv(gw_fts_high_path)
background_feats=pd.read_csv(background_fts_path)
#background_feats_high=pd.read_csv(background_fts_high_path)
background_feats = shuffle(background_feats, random_state=42)
background_feats.reset_index(drop=True, inplace=True)
background_feats['label']=0
relayed_feats = shuffle(relayed_feats, random_state=42)
relayed_feats.reset_index(drop=True, inplace=True)
relayed_feats['label']=1
train=pd.concat([relayed_feats,background_feats],ignore_index=True)


#[ag_res1, ag_res2, fimp1, fimp2, cmatrix, perf, aucscore] = main_ag(train, test, "target")
#ff = open("./BinaryTraining.score", "w+")
#output_avg(foldtotal, ag_res1, ag_res2, fimp1, fimp2, cmatrix, perf, aucscore, ff)

In [6]:


# Training binary classifiers: 8 base models, 2 DL models
predictor = train_main(train, "label")
predstack = train_multilayerstacking(train, "label")





Verbosity: 4 (Maximum Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
GPU Count:          1
Memory Avail:       11.13 GB / 12.67 GB (87.8%)
Disk Space Avail:   79.12 GB / 112.64 GB (70.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='high'         : Strong accuracy wi

[1]	valid_set's binary_logloss: 0.146551	valid_set's balanced_accuracy: 0.5
[2]	valid_set's binary_logloss: 0.131447	valid_set's balanced_accuracy: 0.5
[3]	valid_set's binary_logloss: 0.120149	valid_set's balanced_accuracy: 0.5
[4]	valid_set's binary_logloss: 0.110666	valid_set's balanced_accuracy: 0.5
[5]	valid_set's binary_logloss: 0.103796	valid_set's balanced_accuracy: 0.5
[6]	valid_set's binary_logloss: 0.0971814	valid_set's balanced_accuracy: 0.5
[7]	valid_set's binary_logloss: 0.0922312	valid_set's balanced_accuracy: 0.5
[8]	valid_set's binary_logloss: 0.0874355	valid_set's balanced_accuracy: 0.5
[9]	valid_set's binary_logloss: 0.083236	valid_set's balanced_accuracy: 0.5
[10]	valid_set's binary_logloss: 0.078771	valid_set's balanced_accuracy: 0.5
[11]	valid_set's binary_logloss: 0.075347	valid_set's balanced_accuracy: 0.5
[12]	valid_set's binary_logloss: 0.0720623	valid_set's balanced_accuracy: 0.919192
[13]	valid_set's binary_logloss: 0.0692888	valid_set's balanced_accuracy: 0.

Saving /content/AGmodels/models/LightGBMXT/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMXT/y_pred_proba_val.pkl
	0.9596	 = Validation score   (balanced_accuracy)
	4.24s	 = Training   runtime
	0.02s	 = Validation runtime
	90119.2	 = Inference  throughput (rows/s | 2132 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: LightGBM ...
	Fitting LightGBM with 'num_gpus': 0, 'num_cpus': 1
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05}


[265]	valid_set's binary_logloss: 0.0166692	valid_set's balanced_accuracy: 0.959596
[266]	valid_set's binary_logloss: 0.0166249	valid_set's balanced_accuracy: 0.959596
[267]	valid_set's binary_logloss: 0.0166546	valid_set's balanced_accuracy: 0.959596
[268]	valid_set's binary_logloss: 0.0166579	valid_set's balanced_accuracy: 0.959596
[269]	valid_set's binary_logloss: 0.0166437	valid_set's balanced_accuracy: 0.959596
[1]	valid_set's binary_logloss: 0.143261	valid_set's balanced_accuracy: 0.5
[2]	valid_set's binary_logloss: 0.12659	valid_set's balanced_accuracy: 0.5
[3]	valid_set's binary_logloss: 0.114873	valid_set's balanced_accuracy: 0.5
[4]	valid_set's binary_logloss: 0.105623	valid_set's balanced_accuracy: 0.5
[5]	valid_set's binary_logloss: 0.0979141	valid_set's balanced_accuracy: 0.5
[6]	valid_set's binary_logloss: 0.0914647	valid_set's balanced_accuracy: 0.5
[7]	valid_set's binary_logloss: 0.08588	valid_set's balanced_accuracy: 0.5
[8]	valid_set's binary_logloss: 0.0808704	valid_

Saving /content/AGmodels/models/LightGBM/model.pkl
Saving /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
	0.9596	 = Validation score   (balanced_accuracy)
	1.49s	 = Training   runtime
	0.01s	 = Validation runtime
	308993.0	 = Inference  throughput (rows/s | 2132 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: RandomForestGini ...
	Fitting RandomForestGini with 'num_gpus': 0, 'num_cpus': 2


...

Saving /content/AGmodels/models/XGBoost/model.pkl
Saving /content/AGmodels/utils/attr/XGBoost/y_pred_proba_val.pkl
	0.9596	 = Validation score   (balanced_accuracy)
	2.24s	 = Training   runtime
	0.01s	 = Validation runtime
	151864.8	 = Inference  throughput (rows/s | 2132 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: NeuralNetTorch ...
	Fitting NeuralNetTorch with 'num_gpus': 0, 'num_cpus': 1
Tabular Neural Network treats features as the following types:
{
    "continuous": [
        "avg_in",
        "avg_out",
        "std_order_out",
        "pcap_nb"
    ],
    "skewed": [
        "pkts_rate",
        "duration",
        "mean_total_pkts",
        "median_total_pkts",
        "mode_total_pkts",
        "mean_bytes_sent",
        "median_bytes_sent",
        "mode_bytes_sent",
        "mean_bytes_recv",
        "median_bytes_recv",
        "mode_bytes_recv",
        "gap_between_conns",
        "max_in",
        "max_out",
        "max_total",
        "std_i

[1]	valid_set's binary_logloss: 0.160815	valid_set's balanced_accuracy: 0.5
[2]	valid_set's binary_logloss: 0.146148	valid_set's balanced_accuracy: 0.5
[3]	valid_set's binary_logloss: 0.135539	valid_set's balanced_accuracy: 0.5
[4]	valid_set's binary_logloss: 0.127111	valid_set's balanced_accuracy: 0.5
[5]	valid_set's binary_logloss: 0.119932	valid_set's balanced_accuracy: 0.5
[6]	valid_set's binary_logloss: 0.113538	valid_set's balanced_accuracy: 0.5
[7]	valid_set's binary_logloss: 0.108158	valid_set's balanced_accuracy: 0.5
[8]	valid_set's binary_logloss: 0.103397	valid_set's balanced_accuracy: 0.5
[9]	valid_set's binary_logloss: 0.0991391	valid_set's balanced_accuracy: 0.5
[10]	valid_set's binary_logloss: 0.0951331	valid_set's balanced_accuracy: 0.5
[11]	valid_set's binary_logloss: 0.0915237	valid_set's balanced_accuracy: 0.5
[12]	valid_set's binary_logloss: 0.0881382	valid_set's balanced_accuracy: 0.5
[13]	valid_set's binary_logloss: 0.0850498	valid_set's balanced_accuracy: 0.5
[14

Saving /content/AGmodels/models/LightGBMLarge/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMLarge/y_pred_proba_val.pkl
	0.9596	 = Validation score   (balanced_accuracy)
	1.56s	 = Training   runtime
	0.01s	 = Validation runtime
	372314.8	 = Inference  throughput (rows/s | 2132 batch size)
Saving /content/AGmodels/models/trainer.pkl
Loading: /content/AGmodels/utils/attr/CatBoost/y_pred_proba_val.pkl


[153]	valid_set's binary_logloss: 0.0240941	valid_set's balanced_accuracy: 0.959104
[154]	valid_set's binary_logloss: 0.0241679	valid_set's balanced_accuracy: 0.959104
[155]	valid_set's binary_logloss: 0.0242424	valid_set's balanced_accuracy: 0.959104
[156]	valid_set's binary_logloss: 0.0242723	valid_set's balanced_accuracy: 0.959104
[157]	valid_set's binary_logloss: 0.0243324	valid_set's balanced_accuracy: 0.959104
[158]	valid_set's binary_logloss: 0.0243917	valid_set's balanced_accuracy: 0.959104
[159]	valid_set's binary_logloss: 0.0244212	valid_set's balanced_accuracy: 0.959104
[160]	valid_set's binary_logloss: 0.0244773	valid_set's balanced_accuracy: 0.959104
[161]	valid_set's binary_logloss: 0.024532	valid_set's balanced_accuracy: 0.959104
[162]	valid_set's binary_logloss: 0.0245802	valid_set's balanced_accuracy: 0.959104
[163]	valid_set's binary_logloss: 0.0246149	valid_set's balanced_accuracy: 0.959104
[164]	valid_set's binary_logloss: 0.0246511	valid_set's balanced_accuracy: 0.

Loading: /content/AGmodels/utils/attr/ExtraTreesEntr/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/KNeighborsUnif/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/LightGBMLarge/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/NeuralNetFastAI/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/LightGBMXT/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/XGBoost/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/RandomForestEntr/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/RandomForestGini/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/KNeighborsDist/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/NeuralNetTorch/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/ExtraTreesGini/y_pred_proba_val.pkl
Model configs that will be trained (in order):
	WeightedEnsemble_L2: 	{'ag_args': {'valid_base': False, 'name_bag_suffix': ''

In [12]:

relayed_feats_eval_path='/content/features_lim_relayed_eval.csv'
background_feats_eval_path='/content/features_lim_background_eval.csv'
relayed_feats_eval=pd.read_csv(relayed_feats_eval_path)
relayed_feats_eval['label'] = 1

background_feats_eval=pd.read_csv(background_feats_eval_path)
background_feats_eval['label'] = 0
eval_data=pd.concat([relayed_feats_eval,background_feats_eval])
test=eval_data.sample(frac=1, random_state=42)

x_test = test.iloc[:,:-1].copy()
y_test = test.iloc[:,-1].copy()
print("###################~Testing Trained Models ############################")
#res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score = test_main(x_test, y_test, predictor, test, train)
# Uncomment for test results with feature importance (longer run time)
res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score = test_main(x_test, y_test, predictor, test, train, True)

print("####################Stacking & Weighted Ensemble Testing###########################")
#res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_test, y_test, predstack, test, train)
# With feature importance
res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_test, y_test, predstack, test, train, True)

Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl


###################~Testing Trained Models ############################


Loading: /content/AGmodels/models/KNeighborsDist/model.pkl
Loading: /content/AGmodels/models/LightGBMXT/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/RandomForestGini/model.pkl
Loading: /content/AGmodels/models/RandomForestEntr/model.pkl
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/ExtraTreesGini/model.pkl
Loading: /content/AGmodels/models/ExtraTreesEntr/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/XGBoost/model.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl


[*]Model performance breakdown on Test data:
                  model  score_test  score_val        eval_metric  \
0      RandomForestEntr    0.994382   0.959596  balanced_accuracy   
1        ExtraTreesEntr    0.994382   0.959596  balanced_accuracy   
2      RandomForestGini    0.994382   0.959596  balanced_accuracy   
3        ExtraTreesGini    0.994382   0.959596  balanced_accuracy   
4        KNeighborsDist    0.983146   0.964646  balanced_accuracy   
5            LightGBMXT    0.960674   0.959596  balanced_accuracy   
6         LightGBMLarge    0.955056   0.959596  balanced_accuracy   
7              CatBoost    0.955056   0.964646  balanced_accuracy   
8       NeuralNetFastAI    0.955056   0.959350  balanced_accuracy   
9        KNeighborsUnif    0.954876   0.964646  balanced_accuracy   
10  WeightedEnsemble_L2    0.954876   0.964646  balanced_accuracy   
11             LightGBM    0.949438   0.959596  balanced_accuracy   
12              XGBoost    0.949438   0.959596  balanced_a

Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Skipping roc_auc because no prediction probabilities are available to score.
These features in provided data are not utilized by the predictor and will be ignored: ['avg_total', 'nb_pkts_total', 'med_per_sec', 'min_per_sec', 'sum_number_pkts']
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Computing feature importance via permutation shuffling for 47 features using 5000 rows with 5 shuffle sets...
Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl


...

[*]Predictions:  354     0
2305    0
672     0
562     0
2585    0
       ..
1549    0
1006    0
1041    0
1205    0
771     0
Name: label, Length: 2870, dtype: int64


Loading: /content/AGmodels/stacked/models/KNeighborsDist_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMXT_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBM_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/RandomForestGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/RandomForestEntr_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/CatBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/ExtraTreesGini_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/ExtraTreesEntr_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetFastAI_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/NeuralNetTorch_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/LightGBMLarge_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
These features in provided data are not utilized by the pre

$$$$$$$$ RESULT STACKING $$$$$$$$
                       model  score_test  score_val        eval_metric  \
0         LightGBMXT_BAG_L1    0.998921   0.943809  balanced_accuracy   
1           LightGBM_BAG_L1    0.998202   0.946818  balanced_accuracy   
2            XGBoost_BAG_L1    0.997123   0.947706  balanced_accuracy   
3       WeightedEnsemble_L2    0.997123   0.947706  balanced_accuracy   
4   RandomForestGini_BAG_L1    0.994606   0.939283  balanced_accuracy   
5      LightGBMLarge_BAG_L1    0.994247   0.944722  balanced_accuracy   
6     ExtraTreesEntr_BAG_L1    0.994067   0.939308  balanced_accuracy   
7     KNeighborsDist_BAG_L1    0.993887   0.938076  balanced_accuracy   
8   RandomForestEntr_BAG_L1    0.993168   0.937791  balanced_accuracy   
9     ExtraTreesGini_BAG_L1    0.991190   0.938827  balanced_accuracy   
10    KNeighborsUnif_BAG_L1    0.990471   0.937619  balanced_accuracy   
11   NeuralNetFastAI_BAG_L1    0.976449   0.938125  balanced_accuracy   
12    NeuralNetT

Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
	87.43s	= Expected runtime (17.49s per shuffle set)
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/stacked/models/XGBoost_BAG_L1/model.pkl
Loading: /content/AGmodels/stacked/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/stacked/models/

Feature Importance on test data:                         importance    stddev       p_value  n  p99_high  \
mean_bytes_sent          0.363638  0.011839  1.346540e-07  5  0.388015   
75th_percentile_in       0.031636  0.010539  1.282096e-03  5  0.053334   
gap_between_conns        0.021188  0.005020  3.514900e-04  5  0.031525   
pcap_nb                  0.012151  0.003347  6.263943e-04  5  0.019043   
pkts_rate                0.010867  0.004306  2.427806e-03  5  0.019733   
mode_total_pkts          0.008473  0.002756  1.172163e-03  5  0.014147   
mean_total_pkts          0.007913  0.004017  5.822526e-03  5  0.016184   
median_total_pkts        0.005419  0.001827  1.341359e-03  5  0.009181   
std_in                   0.005111  0.002548  5.468199e-03  5  0.010357   
max_in                   0.002548  0.001768  1.611049e-02  5  0.006188   
sum_intertimestats       0.002102  0.001535  1.876967e-02  5  0.005263   
max_total                0.001666  0.000937  8.236183e-03  5  0.003595   
conn

In [11]:
ff = open("./BinaryTraining.score", "w+")

#ff = open("./BinaryTraining.score", "w+")
#output_avg(foldtotal, ag_res1, ag_res2, fimp1, fimp2, cmatrix, perf, aucscore, ff)

output_avg(foldtotal, res1, res2, fimp1, fimp2, cmatrix, perf,auc_score, ff)

[2780, 1, 8, 81]
